<!-- Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. -->

# MJX 01 — Concepts & MJCF Models

Goal: build a mental model of MuJoCo.

- **`mjModel`**: the *static* description of the world, compiled from an MJCF (XML) file. Bodies, joints, geoms, actuators, masses, etc.
- **`mjData`**: the *dynamic* state that changes every step — `qpos` (positions), `qvel` (velocities), `ctrl` (control inputs), `xpos` (world positions), and more.
- **`mj_step`**: advance the physics by one timestep. **`mj_forward`**: recompute derived quantities without integrating time.

We build a minimal pendulum directly from an MJCF string, simulate it, plot its state, and render a short video — all headless on the GPU via EGL.

In [ ]:
import os
# Headless offscreen rendering backend (set before importing mujoco)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import matplotlib.pyplot as plt
import mujoco
import imageio

In [ ]:
# A minimal MJCF: a floor + a single-hinge pendulum that swings under gravity.
MJCF = """
<mujoco model="pendulum">
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="2 2 0.1" rgba="0.8 0.9 0.8 1"/>
    <body name="pole" pos="0 0 1.2">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom name="mass" type="capsule" fromto="0 0 0 0 0 -0.6" size="0.05" rgba="0.2 0.4 0.9 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(MJCF)
data = mujoco.MjData(model)

print("nq (generalized positions):", model.nq)
print("nv (generalized velocities):", model.nv)
print("nbody:", model.nbody)
print("timestep:", model.opt.timestep)
print("joint name:", mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, 0))

In [ ]:
# Simulate: start the pendulum from a horizontal angle and let it swing.
mujoco.mj_resetData(model, data)
data.qpos[0] = np.pi / 2  # initial hinge angle

times, angles = [], []
for _ in range(1500):
    mujoco.mj_step(model, data)
    times.append(data.time)
    angles.append(data.qpos[0])

plt.figure(figsize=(7, 3))
plt.plot(times, angles)
plt.xlabel("time (s)")
plt.ylabel("hinge angle (rad)")
plt.title("Pendulum qpos over time")
plt.grid(True)
plt.show()

In [ ]:
# Render a short video of the swing (offscreen EGL).
os.makedirs("output/videos", exist_ok=True)
renderer = mujoco.Renderer(model, height=320, width=480)

mujoco.mj_resetData(model, data)
data.qpos[0] = np.pi / 2

frames = []
for i in range(600):
    mujoco.mj_step(model, data)
    if i % 4 == 0:  # ~125 fps sim -> ~30 fps video
        renderer.update_scene(data)
        frames.append(renderer.render())

video_path = "output/videos/mjx01_pendulum.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Saved {len(frames)} frames to {video_path}")

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx01_pendulum.mp4")